<a href="https://colab.research.google.com/github/Eitan177/GOAL_InsilicoControl/blob/main/InSilicoControls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
FILENAME = "pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed"
!wget -O /content/{FILENAME} https://github.com/Eitan177/GOAL_InsilicoControl/raw/main/{FILENAME}

--2026-04-16 19:23:32--  https://github.com/Eitan177/GOAL_InsilicoControl/raw/main/pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Eitan177/GOAL_InsilicoControl/main/pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed [following]
--2026-04-16 19:23:33--  https://raw.githubusercontent.com/Eitan177/GOAL_InsilicoControl/main/pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2709449 (2.6M) [text/plain]
Saving to: ‘/content/pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed’

/content/pennseq.v2 100

In [5]:
!pip install pysam
!apt-get update
!apt-get install -y samtools bedtools
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!gunzip hg38.fa.gz
!samtools faidx hg38.fa
!wget https://hgdownload.cse.ucsc.edu/admin/exe/linux.x86_64/bigBedToBed
!chmod +x bigBedToBed
!mv bigBedToBed /usr/local/bin/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 59.2 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubun

In [6]:
PROBES_BED = "/content/pennseq.v2.all_probes.hg38.exon_intersect_dedup.bed"

# 1. Download MANE bigBed, convert to BED12, and extract individual exons
!echo "Downloading MANE annotation..."
!wget -q -O MANE.bb https://ftp.ncbi.nlm.nih.gov/refseq/MANE/trackhub/data/release_1.0/MANE.GRCh38.v1.0.refseq.bb
!echo "Converting bigBed and extracting exons..."
!bigBedToBed MANE.bb MANE.bed12

# Parse BED12 into individual exons using native Python, extracting CDS and intron info
with open('MANE.bed12', 'r') as infile, open('hg38_exons_raw.bed', 'w') as outfile:
    for line in infile:
        cols = line.strip().split()
        if len(cols) < 12: continue
        chrom, chromStart = cols[0], int(cols[1])
        thickStart, thickEnd = int(cols[6]), int(cols[7])
        sizes = [int(x) for x in cols[10].strip(',').split(',')]
        starts = [int(x) for x in cols[11].strip(',').split(',')]

        num_exons = len(sizes)
        for i in range(num_exons):
            exon_start = chromStart + starts[i]
            exon_end = exon_start + sizes[i]

            # Check if this exon has any coding sequence (CDS)
            if exon_end > thickStart and exon_start < thickEnd:
                cds_start = max(exon_start, thickStart)
                cds_end = min(exon_end, thickEnd)

                # Exons only have left/right introns if they are not the first/last exon respectively
                has_left_intron = (i > 0)
                has_right_intron = (i < num_exons - 1)

                outfile.write(f"{chrom}\t{exon_start}\t{exon_end}\t{cds_start}\t{cds_end}\t{has_left_intron}\t{has_right_intron}\n")

# Sort exons to get a clean set of coordinates (we skip merge to retain our new columns)
!bedtools sort -i hg38_exons_raw.bed > hg38_exons.bed

# 2. Analyze exon and probe coverage
!echo "Merging contiguous probes..."
!bedtools sort -i {PROBES_BED} | bedtools merge -i - > merged_probes.bed

!echo "Finding covered exons and unused probes..."
!bedtools intersect -a hg38_exons.bed -b merged_probes.bed -f 0.95 -wa -u > fully_covered_exons.bed
!bedtools intersect -a hg38_exons.bed -b merged_probes.bed -wa -u > any_covered_exons.bed
!bedtools intersect -a any_covered_exons.bed -b fully_covered_exons.bed -v > partially_covered_exons.bed
!bedtools intersect -a {PROBES_BED} -b hg38_exons.bed -v > probes_without_exons.bed
!bedtools intersect -a merged_probes.bed -b any_covered_exons.bed -v > unused_probes.bed

# Print Exon Statistics
!FULL=$(wc -l < fully_covered_exons.bed); PARTIAL=$(wc -l < partially_covered_exons.bed); OFF_TARGET=$(wc -l < probes_without_exons.bed); UNUSED=$(wc -l < unused_probes.bed); \
echo -e "\n============================================="; \
echo "              COVERAGE SUMMARY               "; \
echo "============================================="; \
echo "Exons with >95% coverage (USED):    $FULL"; \
echo "Exons with partial coverage (USED): $PARTIAL"; \
echo "Probes with no exon coverage:       $OFF_TARGET"; \
echo "Unused contiguous probes (ADDED):   $UNUSED"; \
echo "============================================="

Converting bigBed and extracting exons...
Merging contiguous probes...
Finding covered exons and unused probes...

              COVERAGE SUMMARY               
Exons with >95% coverage (USED):    6741
Exons with partial coverage (USED): 1274
Probes with no exon coverage:       8781
Unused contiguous probes (ADDED):   5147


In [7]:
%%writefile generate_target_snvs.py
import random
import sys

fully_covered_bed = sys.argv[1]
partially_covered_bed = sys.argv[2]
unused_probes_bed = sys.argv[3]
output_bed = sys.argv[4]

def process_exons(bed_file, outfile, label_prefix):
    with open(bed_file, 'r') as infile:
        for idx, line in enumerate(infile):
            if not line.strip(): continue
            parts = line.strip().split('\t')
            chrom = parts[0]
            start = int(parts[1])
            end = int(parts[2])
            cds_start = int(parts[3])
            cds_end = int(parts[4])
            has_left_intron = parts[5] == 'True'
            has_right_intron = parts[6] == 'True'

            exon_label = f"{label_prefix}_{idx+1}_{chrom}_{start}_{end}"

            # Left intronic side (only if an intron actually exists here)
            if has_left_intron:
                left_snv = max(0, start - random.randint(1, 10))
                outfile.write(f"{chrom}\t{left_snv}\t{left_snv+1}\t{exon_label}_left_intron\n")

            # Exonic (strictly bounded within the CDS part of the exon)
            if cds_end > cds_start:
                exon_snv = random.randint(cds_start, cds_end - 1)
                outfile.write(f"{chrom}\t{exon_snv}\t{exon_snv+1}\t{exon_label}_cds\n")

            # Right intronic side (only if an intron actually exists here)
            if has_right_intron:
                right_snv = end + random.randint(1, 10)
                outfile.write(f"{chrom}\t{right_snv}\t{right_snv+1}\t{exon_label}_right_intron\n")

with open(output_bed, 'w') as outfile:
    # Process both fully and partially covered exons
    process_exons(fully_covered_bed, outfile, "FullyCovered")
    process_exons(partially_covered_bed, outfile, "PartiallyCovered")

    # Process unused probes (1 variant per probe)
    with open(unused_probes_bed, 'r') as infile:
        for idx, line in enumerate(infile):
            if not line.strip(): continue
            parts = line.strip().split('\t')
            chrom = parts[0]
            start = int(parts[1])
            end = int(parts[2])

            # Calculate exact midpoint of the probe
            midpoint = (start + end) // 2

            probe_label = f"UnusedProbe_{idx+1}_{chrom}_{start}_{end}"
            outfile.write(f"{chrom}\t{midpoint}\t{midpoint+1}\t{probe_label}_midpoint\n")


Writing generate_target_snvs.py


In [8]:
!python generate_target_snvs.py fully_covered_exons.bed partially_covered_exons.bed unused_probes.bed igv_variant_navigator.bed

# Print Variant Statistics
!VARIANTS=$(wc -l < igv_variant_navigator.bed); \
echo -e "\n============================================="; \
echo "              VARIANT SUMMARY                "; \
echo "============================================="; \
echo "Total SNVs generated for BAM:       $VARIANTS"; \
echo "============================================="; \
echo "An 'igv_variant_navigator.bed' file has been created to view these in IGV."



              VARIANT SUMMARY                
Total SNVs generated for BAM:       28148
An 'igv_variant_navigator.bed' file has been created to view these in IGV.


In [9]:
%%writefile generate_synthetic_bam.py
import pysam
import argparse
import random
from tqdm import tqdm
from collections import defaultdict

def apply_mutations(read_start, read_length, ref_fasta, chrom, variants_to_apply):
    """
    Safely applies SNVs, Insertions, and Deletions to a read sequence,
    returning the modified sequence string, CIGAR tuples, and NM score.
    """
    try:
        # Fetch slightly more reference sequence to handle deletions
        ref_seq = ref_fasta.fetch(chrom, read_start, read_start + read_length + 50).upper()
    except Exception:
        return None, None, 0

    cigar = []
    out_seq = []
    nm = 0
    ref_idx = 0
    read_idx = 0

    variants_to_apply.sort(key=lambda x: x['pos'])

    for v in variants_to_apply:
        v_ref_idx = v['pos'] - read_start
        if v_ref_idx < ref_idx: continue

        # 1. Fill matching bases leading up to the variant
        if v_ref_idx > ref_idx:
            dist = v_ref_idx - ref_idx
            dist = min(dist, read_length - read_idx) # Prevent exceeding read length
            if dist > 0:
                out_seq.append(ref_seq[ref_idx:ref_idx+dist])
                cigar.append([0, dist]) # M
                ref_idx += dist
                read_idx += dist

        if read_idx >= read_length: break

        # 2. Apply the variant
        ref_len = len(v['ref'])
        alt_len = len(v['alt'])

        if ref_len == 1 and alt_len == 1:
            # SNV
            out_seq.append(v['alt'])
            cigar.append([0, 1])
            ref_idx += 1
            read_idx += 1
            nm += 1
        elif alt_len > ref_len:
            # Insertion (First base is the anchor)
            ins_seq = v['alt']
            avail = read_length - read_idx
            if avail > 0:
                added_seq = ins_seq[:avail]
                out_seq.append(added_seq)
                cigar.append([0, 1]) # Match for the anchor base
                if len(added_seq) > 1:
                    cigar.append([1, len(added_seq) - 1]) # Insertion for the rest
                ref_idx += 1
                read_idx += len(added_seq)
                nm += len(added_seq) - 1
        elif ref_len > alt_len:
            # Deletion
            out_seq.append(v['alt'])
            cigar.append([0, 1]) # Match for the anchor base
            cigar.append([2, ref_len - 1]) # Deletion
            ref_idx += ref_len
            read_idx += 1
            nm += ref_len - 1

    # 3. Fill the rest of the read with standard reference bases
    if read_idx < read_length:
        rem = read_length - read_idx
        out_seq.append(ref_seq[ref_idx:ref_idx+rem])
        cigar.append([0, rem])

    # Consolidate adjacent CIGAR operations (e.g. merge M operations)
    final_cigar = []
    for op, length in cigar:
        if length > 0:
            if final_cigar and final_cigar[-1][0] == op:
                final_cigar[-1][1] += length
            else:
                final_cigar.append([op, length])

    return "".join(out_seq), [tuple(c) for c in final_cigar], nm

def generate_synthetic_bam(bed_file, fasta_file, output_bam, output_vcf, depth, vaf, rg_id, rg_sm, insert_size, insert_std, indel_interval, read_length=150):
    print(f"Loading reference {fasta_file}...")
    fasta = pysam.FastaFile(fasta_file)

    header = {
        'HD': {'VN': '1.0', 'SO': 'unsorted'},
        'SQ': [{'SN': chrom, 'LN': length} for chrom, length in zip(fasta.references, fasta.lengths)],
        'RG': [{'ID': rg_id, 'SM': rg_sm, 'PL': 'ILLUMINA'}]
    }

    chrom_to_tid = {chrom: i for i, chrom in enumerate(fasta.references)}

    # 1. Read locations, apply indels periodically, and group by parent Exon/Probe
    groups = defaultdict(list)
    variant_count = 0

    with open(bed_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip(): continue
            parts = line.strip().split('\t')
            chrom = parts[0]
            pos = int(parts[1])
            label = parts[3]

            group_key = "_".join(label.split('_')[:5])
            ref_base = fasta.fetch(chrom, pos, pos+1).upper()

            # Decide if this should be an Indel based on the interval
            if indel_interval > 0 and (variant_count + 1) % indel_interval == 0:
                indel_len = random.randint(1, 4)
                if random.choice(['INS', 'DEL']) == 'INS':
                    ins_bases = "".join(random.choices(['A', 'C', 'G', 'T'], k=indel_len))
                    ref_seq = ref_base
                    alt_seq = ref_base + ins_bases
                else: # DEL
                    ref_seq = fasta.fetch(chrom, pos, pos + indel_len + 1).upper()
                    alt_seq = ref_base
            else: # Standard SNV
                alt_bases = [b for b in ['A', 'C', 'G', 'T'] if b != ref_base]
                alt_seq = random.choice(alt_bases) if alt_bases else 'A'
                ref_seq = ref_base

            groups[group_key].append({
                'chrom': chrom,
                'pos': pos,
                'vcf_pos': pos + 1,
                'ref': ref_seq,
                'alt': alt_seq
            })
            variant_count += 1

    print(f"Generating single-distribution read clusters across {len(groups)} regions...")

    with pysam.AlignmentFile(output_bam, "wb", header=header) as out_bam, open(output_vcf, "w") as vcf:
        vcf.write("##fileformat=VCFv4.2\n")
        vcf.write('##INFO=<ID=DP,Number=1,Type=Integer,Description="Target Depth">\n')
        vcf.write('##INFO=<ID=AF,Number=A,Type=Float,Description="Target Allele Frequency">\n')
        vcf.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")

        pbar = tqdm(total=len(groups), desc="Processing Regions")

        for group_key, variants in groups.items():
            chrom = variants[0]['chrom']

            if chrom not in fasta.references and f"chr{chrom}" in fasta.references:
                chrom = f"chr{chrom}"
                for v in variants: v['chrom'] = chrom
            if chrom not in fasta.references:
                pbar.update(1)
                continue

            tid = chrom_to_tid[chrom]
            min_pos = min(v['pos'] for v in variants)
            max_pos = max(v['pos'] for v in variants)

            variant_coverage = {v['pos']: 0 for v in variants}
            generated_pairs = []
            max_pairs = depth * 500

            # 2. Generate fragment pairs until minimum target edge depth is met
            while len(generated_pairs) < max_pairs:
                if all(cov >= depth for cov in variant_coverage.values()): break

                # FIX: Uniformly distribute fragments to completely eliminate spatial strand bias
                # The range is buffered by insert_size to ensure both fwd and rev reads can reach the outer variants
                frag_center = random.randint(min_pos - insert_size, max_pos + insert_size)

                tlen = int(random.gauss(insert_size, insert_std))
                tlen = max(read_length + 10, tlen) # Prevent physically impossible inserts

                frag_start = int(frag_center - tlen / 2)
                frag_end = frag_start + tlen
                fwd_start = frag_start
                fwd_end = fwd_start + read_length
                rev_start = frag_end - read_length
                rev_end = frag_end

                covers_any = False
                for v in variants:
                    pos = v['pos']
                    if (fwd_start <= pos < fwd_end) or (rev_start <= pos < rev_end):
                        variant_coverage[pos] += 1
                        covers_any = True

                if covers_any or (min_pos - tlen <= frag_center <= max_pos + tlen):
                    generated_pairs.append({
                        'fwd_start': fwd_start,
                        'rev_start': rev_start,
                        'tlen': tlen,
                        'alt_positions': set()
                    })

            # 3. Accurately assign ALT alleles to achieve the requested VAF (Zero Strand Bias)
            for v in variants:
                pos = v['pos']
                covering_pairs = []
                fwd_covers = []
                rev_covers = []
                both_covers = []

                # Group reads by the strand they cover the variant on
                for pair in generated_pairs:
                    covers_fwd = (pair['fwd_start'] <= pos < pair['fwd_start'] + read_length)
                    covers_rev = (pair['rev_start'] <= pos < pair['rev_start'] + read_length)

                    if covers_fwd or covers_rev:
                        covering_pairs.append(pair)

                    if covers_fwd and covers_rev:
                        both_covers.append(pair)
                    elif covers_fwd:
                        fwd_covers.append(pair)
                    elif covers_rev:
                        rev_covers.append(pair)

                if len(covering_pairs) == 0: continue

                num_alts = int(len(covering_pairs) * vaf)
                alt_pairs = []

                # FIX: Stratified Sampling to enforce 50/50 ALT strand balance
                # Step 3A: Sample overlapping pairs first (these inherently have no strand bias)
                num_both = min(len(both_covers), int(num_alts * (len(both_covers) / len(covering_pairs))))
                if num_both > 0:
                    alt_pairs.extend(random.sample(both_covers, num_both))

                # Step 3B: Split the remaining ALT alleles evenly between forward-only and reverse-only pools
                rem_alts = num_alts - len(alt_pairs)
                half_rem = rem_alts // 2

                fwd_to_take = min(len(fwd_covers), half_rem)
                rev_to_take = min(len(rev_covers), half_rem)

                shortfall = rem_alts - (fwd_to_take + rev_to_take)

                fwd_chosen = random.sample(fwd_covers, fwd_to_take)
                rev_chosen = random.sample(rev_covers, rev_to_take)
                alt_pairs.extend(fwd_chosen + rev_chosen)

                # Step 3C: Fill any shortfall if one strand pool happened to be slightly exhausted
                if shortfall > 0:
                    rem_fwd = [p for p in fwd_covers if p not in fwd_chosen]
                    rem_rev = [p for p in rev_covers if p not in rev_chosen]
                    pool = rem_fwd + rem_rev
                    alt_pairs.extend(random.sample(pool, min(shortfall, len(pool))))

                for pair in alt_pairs:
                    pair['alt_positions'].add(pos)

                vcf.write(f"{chrom}\t{v['vcf_pos']}\t.\t{v['ref']}\t{v['alt']}\t.\tPASS\tDP={len(covering_pairs)};AF={vaf}\n")

            # 4. Write the final assembled read pairs using the CIGAR builder
            for i, pair in enumerate(generated_pairs):
                fwd_start = pair['fwd_start']
                rev_start = pair['rev_start']
                tlen = pair['tlen']

                active_variants = [v for v in variants if v['pos'] in pair['alt_positions']]

                fwd_seq, fwd_cigar, fwd_nm = apply_mutations(fwd_start, read_length, fasta, chrom, active_variants)
                rev_seq, rev_cigar, rev_nm = apply_mutations(rev_start, read_length, fasta, chrom, active_variants)

                if not fwd_seq or not rev_seq: continue

                read_name = f"A00979:882:H7JWLDRX7:1:synth_{group_key}:{i}"

                a_fwd = pysam.AlignedSegment()
                a_fwd.query_name = read_name
                a_fwd.query_sequence = fwd_seq
                a_fwd.query_qualities = pysam.qualitystring_to_array("I" * read_length)
                a_fwd.reference_id = tid
                a_fwd.reference_start = fwd_start
                a_fwd.mapping_quality = 60
                a_fwd.cigartuples = fwd_cigar
                a_fwd.next_reference_id = tid
                a_fwd.next_reference_start = rev_start
                a_fwd.template_length = tlen
                a_fwd.set_tags([('RG', rg_id), ('MC', f'{read_length}M'), ('AS', read_length), ('NM', fwd_nm)])

                a_rev = pysam.AlignedSegment()
                a_rev.query_name = read_name
                a_rev.query_sequence = rev_seq
                a_rev.query_qualities = pysam.qualitystring_to_array("I" * read_length)
                a_rev.reference_id = tid
                a_rev.reference_start = rev_start
                a_rev.mapping_quality = 60
                a_rev.cigartuples = rev_cigar
                a_rev.next_reference_id = tid
                a_rev.next_reference_start = fwd_start
                a_rev.template_length = -tlen
                a_rev.set_tags([('RG', rg_id), ('MC', f'{read_length}M'), ('AS', read_length), ('NM', rev_nm)])

                if random.choice([True, False]):
                    a_fwd.flag = 99  # R1 fwd
                    a_rev.flag = 147 # R2 rev
                else:
                    a_fwd.flag = 163 # R2 fwd
                    a_rev.flag = 83  # R1 rev

                out_bam.write(a_fwd)
                out_bam.write(a_rev)

            pbar.update(1)

        pbar.close()

    print(f"Done! Synthetic BAM saved to {output_bam}")
    print(f"Done! Synthetic VCF saved to {output_vcf}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-b", "--bed", required=True)
    parser.add_argument("-r", "--ref", required=True)
    parser.add_argument("-o", "--output", required=True)
    parser.add_argument("--vcf", required=True, help="Output VCF file")
    parser.add_argument("-d", "--depth", type=int, default=100)
    parser.add_argument("-v", "--vaf", type=float, default=0.20)
    parser.add_argument("--rg-id", default="CPDV2510843-SEQ-251103")
    parser.add_argument("--rg-sm", default="CPDV2510843-SEQ-251103")
    parser.add_argument("--insert-size", type=int, default=379, help="Average insert size for synthetic pairs")
    parser.add_argument("--insert-std", type=int, default=20, help="Standard deviation of insert size variation")
    parser.add_argument("--indel-interval", type=int, default=10, help="Make every Nth variant an Indel (0 to disable)")
    args = parser.parse_args()

    generate_synthetic_bam(args.bed, args.ref, args.output, args.vcf, args.depth, args.vaf, args.rg_id, args.rg_sm, args.insert_size, args.insert_std, args.indel_interval)

Writing generate_synthetic_bam.py


In [ ]:
RG_ID = "ReadID123456"
INSERT_SIZE = 302
INSERT_STD = 40
INDEL_INTERVAL = 50

# 1. Generate the raw, unsorted BAM using the generated SNV targets and output a VCF
!python generate_synthetic_bam.py -b igv_variant_navigator.bed -r hg38.fa -o synthetic_spike_unsorted.bam --vcf synthetic_variants.vcf -d 250 -v 0.20 --rg-id {RG_ID} --rg-sm {RG_ID} --insert-size {INSERT_SIZE} --insert-std {INSERT_STD} --indel-interval {INDEL_INTERVAL}

# Clean up any residual temporary files from previous failed runs
!rm -f synthetic_spike.bam.tmp.*

# 2. Sort the BAM file by genomic coordinates
!samtools sort synthetic_spike_unsorted.bam -o synthetic_spike.bam

# 3. Index the sorted BAM file
!samtools index synthetic_spike.bam

print("Sorting and Indexing Complete!")


Loading reference hg38.fa...
Generating single-distribution read clusters across 13162 regions...
Processing Regions: 100% 13162/13162 [03:50<00:00, 57.20it/s]
Done! Synthetic BAM saved to synthetic_spike_unsorted.bam
Done! Synthetic VCF saved to synthetic_variants.vcf
[bam_sort_core] merging from 5 files and 1 in-memory blocks...
Sorting and Indexing Complete!
